<a href="https://colab.research.google.com/github/Marcin19721205/Timeseries_Data_Processing_Basic/blob/main/LSTM_Modelowa_i_Rezeczywista01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

w idealnie mieszanym zbiorniku (bez strat ciepła) temperatura wypływu T3 = T_tank zależy od T1,F1,T2,F2 oraz V1, a F3 matematycznie się redukuje w bilansie energii (nie wnosi nowej informacji). Jeśli chcesz, żeby F3 było realnym wejściem, to musisz dodać fizykę, w której F3 wpływa na pomiar/transport – najprościej: bezwładność czujnika na rurze, której stała czasowa zależy od przepływu (większy przepływ → mniejsza bezwładność). Poniżej robię dokładnie to: liczona jest „prawdziwa” temperatura w zbiorniku, a etykieta T3 to temperatura mierzona na wypływie z lagiem zależnym od F3

In [11]:
# Foreword  # generujemy dataset ~1000 próbek dla zbiornika mieszającego: wejścia (T1,F1,T2,F2,F3) -> wyjście (T3_meas) # dataset
# Why  # bilans energii daje T_tank, a F3 robimy istotnym przez model bezwładności czujnika zależny od przepływu F3 # rationale

import numpy as np  # numeryka # np
import pandas as pd  # ramka danych # pd

rng = np.random.default_rng(123)  # ziarno losowości dla powtarzalności # seed

n_points = 1000  # liczba punktów w dataset # length
dt_s = 1.0  # krok czasowy [s] # dt

V1_m3 = 1.0  # objętość zbiornika [m^3] (stała) # tank volume

T_env_C = 20.0  # temperatura otoczenia [°C] (tu nie używamy strat, ale zostawiam na później) # env
sensor_tau0_s = 15.0  # bazowa stała czasowa czujnika przy małym przepływie [s] # tau0
sensor_k_flow = 30.0  # czułość tau na przepływ (większe -> tau szybciej maleje) # k
sensor_tau_min_s = 2.0  # dolne ograniczenie tau, żeby nie było zera # tau_min

def make_piecewise_steps(n, low, high, step_min, step_max, noise_std=0.0, rng=None):  # generator sygnału schodkowego # step gen
    rng = np.random.default_rng() if rng is None else rng  # RNG # rng
    x = np.zeros(n, dtype=np.float64)  # bufor sygnału # buffer
    t = 0  # indeks czasu # idx
    current = float(rng.uniform(low, high))  # wartość startowa # start
    while t < n:  # do końca szeregu # loop
        step_len = int(rng.integers(step_min, step_max + 1))  # długość utrzymania wartości # step length
        t_end = min(n, t + step_len)  # koniec segmentu # end
        x[t:t_end] = current  # wypełniamy segment # fill
        current = float(rng.uniform(low, high))  # nowa wartość schodka # new level
        t = t_end  # przesuwamy indeks # advance
    if noise_std > 0.0:  # jeśli chcemy szum # noise
        x = x + rng.normal(0.0, noise_std, size=n)  # dodajemy szum # add noise
    return x  # zwrot # return

t_s = np.arange(n_points, dtype=np.float64) * dt_s  # oś czasu [s] # time

# zmienność

T1_C = make_piecewise_steps(n_points, low=5.0,  high=20.0, step_min=20, step_max=80, noise_std=0.2, rng=rng)  # zimna woda [°C] # T1
T2_C = make_piecewise_steps(n_points, low=10.0, high=125.0, step_min=60, step_max=95, noise_std=0.3, rng=rng)  # ciepła woda [°C] # T2

F1_m3s = make_piecewise_steps(n_points, low=0.002, high=0.020, step_min=15, step_max=60, noise_std=0.0002, rng=rng)  # przepływ zimny [m^3/s] # F1
F2_m3s = make_piecewise_steps(n_points, low=0.002, high=0.040, step_min=15, step_max=60, noise_std=0.0002, rng=rng)  # przepływ ciepły [m^3/s] # F2

F1_m3s = np.clip(F1_m3s, 0.0001, None)  # zabezpieczenie dodatniości # clip
F2_m3s = np.clip(F2_m3s, 0.0001, None)  # zabezpieczenie dodatniości # clip

F3_m3s = F1_m3s + F2_m3s + rng.normal(0.0, 0.0005, size=n_points)  # wypływ ~ suma dopływów + zakłócenie # F3
F3_m3s = np.clip(F3_m3s, 0.0001, None)  # zabezpieczenie dodatniości # clip

T_tank_C = np.zeros(n_points, dtype=np.float64)  # „prawdziwa” temperatura w zbiorniku # tank T
T3_C = np.zeros(n_points, dtype=np.float64)  # temperatura mierzona na wypływie (etykieta) # label

T_tank_C[0] = (F1_m3s[0]*T1_C[0] + F2_m3s[0]*T2_C[0]) / (F1_m3s[0] + F2_m3s[0])  # start jako średnia ważona # init
T3_C[0] = T_tank_C[0] + rng.normal(0.0, 0.05)  # start pomiaru z małym szumem # init meas

for k in range(1, n_points):  # pokazujemy dynamikę w czasie # simulate
    T_prev = T_tank_C[k-1]  # poprzednia temperatura w zbiorniku # prev
    F1_prev = F1_m3s[k-1]  # poprzedni F1 # prev
    F2_prev = F2_m3s[k-1]  # poprzedni F2 # prev
    T1_prev = T1_C[k-1]  # poprzedni T1 # prev
    T2_prev = T2_C[k-1]  # poprzedni T2 # prev

    dTdt = (F1_prev*(T1_prev - T_prev) + F2_prev*(T2_prev - T_prev)) / V1_m3  # bilans energii (idealne mieszanie) # dT/dt
    T_tank_C[k] = T_prev + dt_s * dTdt  # Euler do przodu # euler

    F3_prev = F3_m3s[k-1]  # poprzedni F3 # prev
    sensor_tau_s = sensor_tau0_s / (1.0 + sensor_k_flow * F3_prev)  # tau maleje z przepływem # tau(F)
    sensor_tau_s = max(sensor_tau_s, sensor_tau_min_s)  # ograniczenie od dołu # clamp

    T3_prev = T3_C[k-1]  # poprzedni pomiar # prev
    dT3dt = (T_tank_C[k-1] - T3_prev) / sensor_tau_s  # 1-rzędowy lag czujnika # sensor dyn
    T3_C[k] = T3_prev + dt_s * dT3dt + rng.normal(0.0, 0.05)  # aktualizacja + szum pomiaru # update

tank_raw_df = pd.DataFrame({  # dane surowe (raw) do dalszych etapów: okna, normalizacja, modele # raw df
    "t_s": t_s,  # czas [s] # time
    "T1_C": T1_C,  # temp zimna [°C] # T1
    "F1_m3s": F1_m3s,  # przepływ zimny [m^3/s] # F1
    "T2_C": T2_C,  # temp ciepła [°C] # T2
    "F2_m3s": F2_m3s,  # przepływ ciepły [m^3/s] # F2
    "F3_m3s": F3_m3s,  # przepływ wypływu [m^3/s] # F3
    "T3_C": T3_C,  # etykieta: temp na wypływie (mierzona) [°C] # label
})  # end df # end

print(tank_raw_df.shape)  # kontrola rozmiaru (powinno być (1000, 7)) # shape
print(tank_raw_df.head(3))  # podgląd pierwszych wierszy # head


(1000, 7)
   t_s       T1_C    F1_m3s       T2_C    F2_m3s    F3_m3s       T3_C
0  0.0  15.407254  0.007198  46.424298  0.037481  0.045162  41.396047
1  1.0  15.587610  0.006889  46.143427  0.037160  0.044349  41.374084
2  2.0  15.433943  0.007576  46.274107  0.037246  0.045212  41.355349


In [12]:
# Foreword  # wykres wszystkich zmiennych w Plotly dark: temperatury (lewa oś) + przepływy (prawa oś) # overview plot
# Why  # różne jednostki -> 2 osie Y, inaczej przepływy będą "płaskie" obok temperatur # clarity

import plotly.graph_objects as go  # plotly # go

t = tank_raw_df["t_s"]  # oś czasu # time

fig = go.Figure()  # figura # fig

# --- temperatury (lewa oś Y) # temperatures
fig.add_trace(go.Scatter(x=t, y=tank_raw_df["T1_C"], mode="lines", name="T1_C", yaxis="y"))  # T1 # T1
fig.add_trace(go.Scatter(x=t, y=tank_raw_df["T2_C"], mode="lines", name="T2_C", yaxis="y"))  # T2 # T2
fig.add_trace(go.Scatter(x=t, y=tank_raw_df["T3_C"], mode="lines", name="T3_C", yaxis="y"))  # T3 # T3

# --- przepływy (prawa oś Y) # flows
fig.add_trace(go.Scatter(x=t, y=tank_raw_df["F1_m3s"], mode="lines", name="F1_m3s", yaxis="y2"))  # F1 # F1
fig.add_trace(go.Scatter(x=t, y=tank_raw_df["F2_m3s"], mode="lines", name="F2_m3s", yaxis="y2"))  # F2 # F2
fig.add_trace(go.Scatter(x=t, y=tank_raw_df["F3_m3s"], mode="lines", name="F3_m3s", yaxis="y2"))  # F3 # F3

fig.update_layout(  # layout # layout
    template="plotly_dark",  # dark mode # dark
    title="Tank mixing dataset (raw) | Temperatures & Flows",  # tytuł # title
    xaxis=dict(title="t [s]"),  # oś X # x
    yaxis=dict(title="Temperature [°C]"),  # lewa oś # y
    yaxis2=dict(title="Flow [m³/s]", overlaying="y", side="right"),  # prawa oś # y2
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0)  # legenda u góry # legend
)

fig.show()  # show # show


dataset okienkowy (bez normalizacji)

In [32]:
# Foreword  # KROK 1: budujemy dataset okienkowy dla LSTM: X=(Nw,T_window,D) oraz y=(Nw,) jako T3 w następnym kroku # windows
# Why  # LSTM dostaje sekwencję T_window próbek i uczy się przewidywać kolejną wartość (1-step) # rationale

import numpy as np  # numeryka # np

T_window = 60  # długość okna (zmienisz później) # T
feature_cols = ["T1_C", "F1_m3s", "T2_C", "F2_m3s", "F3_m3s"]  # wejścia # X cols
target_col = "T3_C"  # wyjście # y col

X_raw = tank_raw_df[feature_cols].values.astype(np.float32)  # surowe X # X raw
y_raw = tank_raw_df[target_col].values.astype(np.float32)  # surowe y # y raw

N = len(tank_raw_df)  # liczba próbek # N
Nw = N - T_window  # liczba okien # N windows

X_seq = np.zeros((Nw, T_window, len(feature_cols)), dtype=np.float32)  # okna X # X windows
y_seq = np.zeros((Nw,), dtype=np.float32)  # etykiety y # y windows

for i in range(Nw):  # iteracja po oknach # loop
    X_seq[i] = X_raw[i:i+T_window]  # okno wejść # window X
    y_seq[i] = y_raw[i+T_window]  # target = kolejny punkt T3 # y next

print("X_seq shape:", X_seq.shape, "y_seq shape:", y_seq.shape)  # kontrola wymiarów # shapes
print("First window X_seq[0] last row:", X_seq[0, -1, :])  # ostatni krok pierwszego okna # last step
print("First label y_seq[0]:", float(y_seq[0]))  # etykieta do tego okna # y0

# sanity: czy etykieta jest faktycznie T3 z indeksu T_window? # sanity
print("Check y_raw[T_window]:", float(y_raw[T_window]))  # powinno = y_seq[0] # check


X_seq shape: (940, 60, 5) y_seq shape: (940,)
First window X_seq[0] last row: [8.0609293e+00 4.5387382e-03 4.5712791e+01 3.8340285e-02 4.3993067e-02]
First label y_seq[0]: 41.88514709472656
Check y_raw[T_window]: 41.88514709472656


In [33]:
# Foreword  # KROK 2: dzielimy dataset okienkowy czasowo na train/val/test (bez normalizacji) # split
# Why  # przy szeregach czasowych nie mieszamy przyszłości do przeszłości; split robimy po oknach # rationale

train_frac = 0.60  # udział train # train
val_frac = 0.20  # udział val # val

Nw = X_seq.shape[0]  # liczba okien # Nw
n_train = int(Nw * train_frac)  # liczba okien train # n_train
n_val = int(Nw * val_frac)  # liczba okien val # n_val
n_test = Nw - n_train - n_val  # liczba okien test # n_test

X_train = X_seq[:n_train]  # train X # X_train
y_train = y_seq[:n_train]  # train y # y_train

X_val = X_seq[n_train:n_train + n_val]  # val X # X_val
y_val = y_seq[n_train:n_train + n_val]  # val y # y_val

X_test = X_seq[n_train + n_val:]  # test X # X_test
y_test = y_seq[n_train + n_val:]  # test y # y_test

print("Nw:", Nw, "n_train:", n_train, "n_val:", n_val, "n_test:", n_test)  # sizes # sizes
print("X_train:", X_train.shape, "y_train:", y_train.shape)  # shapes # shapes
print("X_val  :", X_val.shape, "y_val  :", y_val.shape)  # shapes # shapes
print("X_test :", X_test.shape, "y_test :", y_test.shape)  # shapes # shapes

# sanity: granice czasowe (czy split jest ciągły) # sanity
print("train last y:", float(y_train[-1]), "| val first y:", float(y_val[0]))  # boundary # boundary
print("val last y  :", float(y_val[-1]),   "| test first y:", float(y_test[0]))  # boundary # boundary


Nw: 940 n_train: 564 n_val: 188 n_test: 188
X_train: (564, 60, 5) y_train: (564,)
X_val  : (188, 60, 5) y_val  : (188,)
X_test : (188, 60, 5) y_test : (188,)
train last y: 27.114002227783203 | val first y: 27.578327178955078
val last y  : 71.66545867919922 | test first y: 71.3822021484375


In [34]:
# Foreword  # KROK 3: normalizacja (standaryzacja) X i y: fit na TRAIN, potem transform train/val/test # normalize
# Why  # stabilny trening: różne skale cech (°C vs m^3/s) psują uczenie bez standaryzacji # rationale

import numpy as np  # numeryka # np

# --- X: mean/std liczymy po całym TRAIN (po czasie i po oknach) # X scaler
X_train_flat = X_train.reshape(-1, X_train.shape[-1])  # spłaszczamy (n_train*T, D) # flat
X_mean = X_train_flat.mean(axis=0)  # średnie cech # mean
X_std = X_train_flat.std(axis=0) + 1e-8  # odchylenia cech + eps # std

X_train_std = (X_train - X_mean) / X_std  # X train znormalizowane # X train std
X_val_std   = (X_val   - X_mean) / X_std  # X val znormalizowane # X val std
X_test_std  = (X_test  - X_mean) / X_std  # X test znormalizowane # X test std

# --- y: mean/std liczymy tylko na TRAIN # y scaler
y_mean = y_train.mean()  # średnia y # y mean
y_std  = y_train.std() + 1e-8  # std y + eps # y std

y_train_std = (y_train - y_mean) / y_std  # y train std # y train std
y_val_std   = (y_val   - y_mean) / y_std  # y val std # y val std
y_test_std  = (y_test  - y_mean) / y_std  # y test std # y test std

print("X_mean:", X_mean)  # kontrola # check
print("X_std :", X_std)  # kontrola # check
print("y_mean:", float(y_mean), "y_std:", float(y_std))  # kontrola # check

# --- sanity: po standaryzacji TRAIN ma ~0 mean i ~1 std # sanity
X_train_std_flat = X_train_std.reshape(-1, X_train_std.shape[-1])  # flat # flat
print("train X mean approx:", X_train_std_flat.mean(axis=0))  # ~0 # ~0
print("train X std  approx:", X_train_std_flat.std(axis=0))  # ~1 # ~1
print("train y mean approx:", float(y_train_std.mean()))  # ~0 # ~0
print("train y std  approx:", float(y_train_std.std()))  # ~1 # ~1


X_mean: [1.3501085e+01 8.9477180e-03 6.3008163e+01 2.8618515e-02 3.7572373e-02]
X_std : [4.3051186e+00 3.3302864e-03 3.3164738e+01 1.0011030e-02 1.0963910e-02]
y_mean: 52.71881103515625 y_std: 25.542095184326172
train X mean approx: [ 6.1050399e-05  6.0825834e-05  4.1556945e-05  1.1984341e-04
 -3.5126421e-05]
train X std  approx: [1.0000141  0.9999994  1.0000013  1.0000019  0.99999964]
train y mean approx: 8.116376903899436e-08
train y std  approx: 0.9999999403953552


In [35]:
# Foreword  # KROK 4: minimalny model LSTM do 1-step predykcji T3 (na danych znormalizowanych) # LSTM
# Why  # zaczynamy od baseline; dopiero jak działa, dokładamy warstwy/regularizację/tuning # rationale

import tensorflow as tf  # TF/Keras # tf
from tensorflow.keras import Sequential  # model # Sequential
from tensorflow.keras.layers import Input, LSTM, Dense  # warstwy # layers
from tensorflow.keras.optimizers import Adam  # optymalizator # Adam

lstm_units = 32  # hiperparametr do późniejszego tuningu # units
lr = 3e-4  # hiperparametr do tuningu (bez szaleństw) # lr
epochs = 150  # limit epok (i tak zatrzyma EarlyStopping) # epochs
batch_size = 32  # batch # batch

model = Sequential(name="tank_lstm_baseline")  # model # model
model.add(Input(shape=(T_window, X_train_std.shape[-1])))  # (60,5) # input
model.add(LSTM(lstm_units))  # LSTM -> stan końcowy # LSTM
model.add(Dense(1, activation="linear"))  # regresja # out

model.compile(  # compile # compile
    optimizer=Adam(learning_rate=lr, clipnorm=1.0),  # clipnorm stabilizuje RNN # opt
    loss="mse",  # MSE # loss
    metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],  # MAE # mae
)

cb = [  # callbacki # callbacks
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),  # stop # ES
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-5),  # LR down # RLR
]

hist = model.fit(  # trening # fit
    X_train_std, y_train_std,  # train # train
    validation_data=(X_val_std, y_val_std),  # val # val
    epochs=epochs,  # epochs # epochs
    batch_size=batch_size,  # batch # batch
    callbacks=cb,  # callbacks # cb
    verbose=1,  # log # verbose
)


Epoch 1/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - loss: 1.3430 - mae: 1.0810 - val_loss: 0.5555 - val_mae: 0.7063 - learning_rate: 3.0000e-04
Epoch 2/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 1.0183 - mae: 0.9407 - val_loss: 0.4154 - val_mae: 0.6089 - learning_rate: 3.0000e-04
Epoch 3/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.8257 - mae: 0.8432 - val_loss: 0.2907 - val_mae: 0.5064 - learning_rate: 3.0000e-04
Epoch 4/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.5928 - mae: 0.7071 - val_loss: 0.1914 - val_mae: 0.4032 - learning_rate: 3.0000e-04
Epoch 5/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.4038 - mae: 0.5668 - val_loss: 0.1174 - val_mae: 0.2922 - learning_rate: 3.0000e-04
Epoch 6/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.2618 - mae: 0.4438 - val_loss: 0.0923 - val_mae: 0.2295 - learning_rate: 3.0000e-04
Epoch 7/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.1479 - mae: 0.3221 - val_loss: 0.1320 - val_mae: 0.2969 - lear

In [36]:
# Foreword  # KROK 5: test LSTM na oknach: predykcja y_std -> odwrót do °C -> metryki + wykresy # test
# Why  # model uczy się na znormalizowanym y, ale wynik musi być fizyczny (°C) # rationale

import numpy as np  # metryki # np
import plotly.graph_objects as go  # wykresy # go

# --- predykcja na TEST (w skali znormalizowanej) # predict std
y_pred_test_std = model.predict(X_test_std, verbose=0).reshape(-1)  # pred std # pred
y_true_test_std = y_test_std.reshape(-1)  # true std # true

# --- odwrót normalizacji do °C # inverse to °C
y_pred_test_C = y_pred_test_std * y_std + y_mean  # pred °C # pred C
y_true_test_C = y_true_test_std * y_std + y_mean  # true °C (z okien) # true C

# --- czas dla testu (okna zaczynają się w testowym segmencie, ale pierwsza etykieta jest po T_window) # time align
test_start_idx = int(Nw * train_frac) + int(Nw * val_frac)  # start okien testu w X_seq/y_seq # test start (windows)
t_all = tank_raw_df["t_s"].values  # czas całkowity # time all

# y_seq indeksuje "etykiety okien" odpowiadające czasowi t = i+T_window # label time
test_label_start_time_idx = test_start_idx + T_window  # indeks czasu dla pierwszej etykiety testu # t0 label
t_test_pred = t_all[test_label_start_time_idx : test_label_start_time_idx + len(y_pred_test_C)]  # wektor czasu # t pred

# --- metryki w °C # metrics
err = y_true_test_C - y_pred_test_C  # błąd # err
mse = float(np.mean(err**2))  # mse # mse
rmse = float(np.sqrt(mse))  # rmse # rmse
mae = float(np.mean(np.abs(err)))  # mae # mae

print("TEST metrics (°C):")  # header # header
print("MAE :", mae)  # mae # mae
print("RMSE:", rmse)  # rmse # rmse
print("MSE :", mse)  # mse # mse

# --- wykres true vs pred # plot
fig = go.Figure()  # fig # fig
fig.add_trace(go.Scatter(x=t_test_pred, y=y_true_test_C, mode="lines", name="T3_true_C"))  # true # true
fig.add_trace(go.Scatter(x=t_test_pred, y=y_pred_test_C, mode="lines", name="T3_pred_C"))  # pred # pred
fig.update_layout(template="plotly_dark", title="TEST | T3 true vs pred (window-based, 1-step)", xaxis_title="t [s]", yaxis_title="T3 [°C]")  # layout # layout
fig.show()  # show # show

# --- wykres residual # residual
fig_e = go.Figure()  # fig # fig
fig_e.add_trace(go.Scatter(x=t_test_pred, y=err, mode="lines", name="error = true - pred"))  # err # err
fig_e.update_layout(template="plotly_dark", title="TEST | residuals (true - pred)", xaxis_title="t [s]", yaxis_title="error [°C]")  # layout # layout
fig_e.show()  # show # show


TEST metrics (°C):
MAE : 6.564852237701416
RMSE: 8.210812989660152
MSE : 67.41744995117188


In [37]:
# Foreword  # KROK 6: uczymy model na przyroście ΔT3 zamiast na absolutnym T3 # delta learning
# Why  # model nie odpływa do średniej i ma mniejszy bias; fizycznie to jest dyskretny bilans energii # rationale

# --- budujemy nowe etykiety delta na bazie y_seq (które było T3_next) i ostatniej wartości T3 w oknie # delta y
T3_last_in_window = X_seq[:, -1, feature_cols.index("T1_C")*0 + (len(feature_cols)-1)*0]  # placeholder # placeholder


In [38]:
# Foreword  # KROK 6a: delta label = T3(t) - T3(t-1) dla etykiet okien # delta label
# Why  # to jest prosta różnica, bez dokładania T3 do wejść # rationale

y_curr = y_raw[T_window-1:-1].astype(np.float32)  # T3(t-1) dla każdego okna # T3 prev
y_next = y_raw[T_window:].astype(np.float32)      # T3(t)   dla każdego okna # T3 next
y_delta_seq = y_next - y_curr  # ΔT3 # delta

print("y_delta_seq shape:", y_delta_seq.shape)  # kontrola # check
print("sanity delta[0]:", float(y_delta_seq[0]), "=", float(y_raw[T_window] - y_raw[T_window-1]))  # sanity # sanity


y_delta_seq shape: (940,)
sanity delta[0]: 0.019351959228515625 = 0.019351959228515625


In [39]:
# Foreword  # KROK 6b: split czasowy dla delta (te same indeksy co wcześniej) # split delta
# Why  # porównanie apples-to-apples z poprzednim modelem # rationale

y_train_d = y_delta_seq[:n_train]  # train delta # train
y_val_d   = y_delta_seq[n_train:n_train+n_val]  # val delta # val
y_test_d  = y_delta_seq[n_train+n_val:]  # test delta # test

print("y_train_d:", y_train_d.shape, "y_val_d:", y_val_d.shape, "y_test_d:", y_test_d.shape)  # shapes # shapes


y_train_d: (564,) y_val_d: (188,) y_test_d: (188,)


In [40]:
# Foreword  # KROK 6c: normalizacja y_delta (fit na train) # scale delta
# Why  # delta ma inną skalę niż T3; normalizujemy osobno # rationale

y_d_mean = y_train_d.mean()  # mean # mean
y_d_std  = y_train_d.std() + 1e-8  # std # std

y_train_d_std = (y_train_d - y_d_mean) / y_d_std  # train # train
y_val_d_std   = (y_val_d   - y_d_mean) / y_d_std  # val # val
y_test_d_std  = (y_test_d  - y_d_mean) / y_d_std  # test # test

print("delta y mean/std:", float(y_d_mean), float(y_d_std))  # check # check


delta y mean/std: -0.026155661791563034 0.47658655047416687


In [41]:
# Foreword  # KROK 6d: trenujemy identyczny LSTM, ale na y_delta_std # train delta
# Why  # sprawdzamy czy bias znika bez dokładania architektury # rationale

import tensorflow as tf  # TF # tf
from tensorflow.keras import Sequential  # seq # Sequential
from tensorflow.keras.layers import Input, LSTM, Dense  # layers # layers
from tensorflow.keras.optimizers import Adam  # opt # Adam

model_d = Sequential(name="tank_lstm_baseline_delta")  # model # model
model_d.add(Input(shape=(T_window, X_train_std.shape[-1])))  # input # input
model_d.add(LSTM(32))  # LSTM # LSTM
model_d.add(Dense(1, activation="linear"))  # out # out

model_d.compile(optimizer=Adam(learning_rate=3e-4, clipnorm=1.0), loss="mse", metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")])  # compile # compile

cb = [  # callbacks # callbacks
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),  # ES # ES
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-5),  # RLR # RLR
]

hist_d = model_d.fit(  # fit # fit
    X_train_std, y_train_d_std,  # train # train
    validation_data=(X_val_std, y_val_d_std),  # val # val
    epochs=150,  # epochs # epochs
    batch_size=32,  # batch # batch
    callbacks=cb,  # cb # cb
    verbose=1,  # verbose # verbose
)


Epoch 1/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 162ms/step - loss: 1.3770 - mae: 0.7945 - val_loss: 0.4532 - val_mae: 0.5675 - learning_rate: 3.0000e-04
Epoch 2/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 1.0837 - mae: 0.6524 - val_loss: 0.5080 - val_mae: 0.5978 - learning_rate: 3.0000e-04
Epoch 3/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.8355 - mae: 0.5747 - val_loss: 0.5812 - val_mae: 0.6291 - learning_rate: 3.0000e-04
Epoch 4/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.6451 - mae: 0.5093 - val_loss: 0.6728 - val_mae: 0.6565 - learning_rate: 3.0000e-04
Epoch 5/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.5647 - mae: 0.4846 - val_loss: 0.7880 - val_mae: 0.6880 - learning_rate: 3.0000e-04
Epoch 6/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.4654 - mae: 0.4335 - val_loss: 0.8953 - val_mae: 0.7281 - learning_rate: 3.0000e-04
Epoch 7/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - loss: 0.5209 - mae: 0.4716 - val_loss: 1.0240 - val_mae: 0.7942 - lea

In [42]:
# Foreword  # KROK 6e: test delta-model: przewidujemy ΔT3, potem składamy T3_pred w °C # test delta
# Why  # wynik ma być w °C, a delta tylko koryguje stan # rationale

import numpy as np  # np # np
import plotly.graph_objects as go  # plot # go

delta_pred_std = model_d.predict(X_test_std, verbose=0).reshape(-1)  # pred delta std # pred
delta_pred = delta_pred_std * y_d_std + y_d_mean  # pred ΔT3 w °C/krok # inv

# T3_prev dla testu: to jest y_raw[t-1] dla etykiet testowych # T3 prev
test_start_idx = n_train + n_val  # start okien testu w y_delta_seq # start
T3_prev_test = y_raw[(test_start_idx + T_window - 1) : (test_start_idx + T_window - 1 + len(delta_pred))]  # T3(t-1) # prev

T3_pred_C = T3_prev_test + delta_pred  # T3_hat(t) # recon
T3_true_C = y_raw[(test_start_idx + T_window) : (test_start_idx + T_window + len(delta_pred))]  # T3(t) # true

t_all = tank_raw_df["t_s"].values  # time # time
t_test = t_all[(test_start_idx + T_window) : (test_start_idx + T_window + len(delta_pred))]  # time for labels # time

err = T3_true_C - T3_pred_C  # error # err
print("TEST delta-model (°C) | MAE:", float(np.mean(np.abs(err))), "RMSE:", float(np.sqrt(np.mean(err**2))))  # metrics # metrics

fig = go.Figure()  # fig # fig
fig.add_trace(go.Scatter(x=t_test, y=T3_true_C, mode="lines", name="T3_true_C"))  # true # true
fig.add_trace(go.Scatter(x=t_test, y=T3_pred_C, mode="lines", name="T3_pred_C_delta"))  # pred # pred
fig.update_layout(template="plotly_dark", title="TEST | T3 true vs pred (delta-learning)", xaxis_title="t [s]", yaxis_title="T3 [°C]")  # layout # layout
fig.show()  # show # show

fig_e = go.Figure()  # fig # fig
fig_e.add_trace(go.Scatter(x=t_test, y=err, mode="lines", name="error = true - pred"))  # err # err
fig_e.update_layout(template="plotly_dark", title="TEST | residuals (delta-learning)", xaxis_title="t [s]", yaxis_title="error [°C]")  # layout # layout
fig_e.show()  # show # show


TEST delta-model (°C) | MAE: 0.27657318115234375 RMSE: 0.3215518891811371


In [43]:
# Foreword  # testujemy data-leak / bug: (A) permutacja etykiet, (B) zerowanie wejścia, (C) rozwalenie alignmentu okna # leak tests
# Why  # jeśli po tych testach nadal jest "idealnie", to model widzi przyszłość albo robimy błąd w rekonstrukcji # rationale

import numpy as np  # numeryka # np
import tensorflow as tf  # TF # tf
from tensorflow.keras import Sequential  # model # Sequential
from tensorflow.keras.layers import Input, LSTM, Dense  # warstwy # layers
from tensorflow.keras.optimizers import Adam  # optymalizator # Adam

assert "model_d" in globals(), "Brak model_d - najpierw wytrenuj delta-model"  # guard # guard
assert "X_train_std" in globals() and "X_test_std" in globals(), "Brak X_*_std - najpierw zrób normalizację X"  # guard # guard
assert "y_raw" in globals() and "T_window" in globals(), "Brak y_raw/T_window - najpierw zbuduj dataset"  # guard # guard
assert "n_train" in globals() and "n_val" in globals(), "Brak n_train/n_val - potrzebne do indeksów testu"  # guard # guard
assert "y_d_mean" in globals() and "y_d_std" in globals(), "Brak y_d_mean/y_d_std - potrzebne do inverse ΔT3"  # guard # guard

# --- helper: metryki delta-model w °C (rekonstrukcja T3 = T3_prev + ΔT3_pred) # helper
def eval_delta_model_C(model, X_test_std, y_raw, T_window, n_train, n_val, y_d_mean, y_d_std):  # fn # fn
    test_start_idx = int(n_train + n_val)  # start okien testu w y_delta_seq # start
    delta_pred_std = model.predict(X_test_std, verbose=0).reshape(-1)  # pred ΔT3 std # pred
    delta_pred = delta_pred_std * float(y_d_std) + float(y_d_mean)  # inverse ΔT3 do °C/krok # inv
    T3_prev = y_raw[(test_start_idx + T_window - 1) : (test_start_idx + T_window - 1 + len(delta_pred))].astype(np.float32)  # T3(t-1) # prev
    T3_true = y_raw[(test_start_idx + T_window)     : (test_start_idx + T_window     + len(delta_pred))].astype(np.float32)  # T3(t) # true
    T3_pred = T3_prev + delta_pred.astype(np.float32)  # rekonstrukcja T3_hat(t) # recon
    err = T3_true - T3_pred  # błąd # err
    mae = float(np.mean(np.abs(err)))  # MAE # mae
    rmse = float(np.sqrt(np.mean(err**2)))  # RMSE # rmse
    return mae, rmse  # return # return

# --- 0) baseline (aktualny model_d) # baseline
mae0, rmse0 = eval_delta_model_C(model_d, X_test_std, y_raw, T_window, n_train, n_val, y_d_mean, y_d_std)  # baseline metrics # base
print("BASELINE delta-model | MAE [°C]:", mae0, "| RMSE [°C]:", rmse0)  # print # print

# --- A) permutacja etykiet na TRAIN (powinno się posypać) # test A
rng = np.random.default_rng(123)  # RNG # rng
y_train_perm = rng.permutation(y_train_d_std)  # tasujemy etykiety # permute

model_perm = Sequential(name="tank_lstm_delta_perm")  # nowy model # model
model_perm.add(Input(shape=(T_window, X_train_std.shape[-1])))  # input # input
model_perm.add(LSTM(32))  # LSTM # LSTM
model_perm.add(Dense(1, activation="linear"))  # out # out
model_perm.compile(optimizer=Adam(learning_rate=3e-4, clipnorm=1.0), loss="mse")  # compile # compile

cb_perm = [  # callbacks # callbacks
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),  # szybki stop # ES
]
model_perm.fit(  # fit # fit
    X_train_std, y_train_perm,  # train (y permutowane) # train
    validation_data=(X_val_std, y_val_d_std),  # val # val
    epochs=30,  # krótko # epochs
    batch_size=32,  # batch # batch
    callbacks=cb_perm,  # cb # cb
    verbose=0,  # ciszej # quiet
)

maeA, rmseA = eval_delta_model_C(model_perm, X_test_std, y_raw, T_window, n_train, n_val, y_d_mean, y_d_std)  # eval # eval
print("TEST A (permute y_train) | MAE [°C]:", maeA, "| RMSE [°C]:", rmseA)  # print # print

# --- B) zerowanie wejścia na TEST (powinno pogorszyć) # test B
X_test_zero = np.zeros_like(X_test_std)  # zerujemy wejście # zero
maeB, rmseB = eval_delta_model_C(model_d, X_test_zero, y_raw, T_window, n_train, n_val, y_d_mean, y_d_std)  # eval # eval
print("TEST B (X_test = 0) | MAE [°C]:", maeB, "| RMSE [°C]:", rmseB)  # print # print

# --- C) rozwalenie alignmentu w oknie (shift w czasie w ramach okna) # test C
X_test_shift = np.roll(X_test_std, shift=+1, axis=1)  # przesuwamy sekwencję o 1 krok w prawo # roll
X_test_shift[:, 0, :] = 0.0  # pierwszy krok zerujemy (żeby nie było wrap-around) # zero head
maeC, rmseC = eval_delta_model_C(model_d, X_test_shift, y_raw, T_window, n_train, n_val, y_d_mean, y_d_std)  # eval # eval
print("TEST C (shift window by +1) | MAE [°C]:", maeC, "| RMSE [°C]:", rmseC)  # print # print


BASELINE delta-model | MAE [°C]: 0.27657318115234375 | RMSE [°C]: 0.3215518891811371
TEST A (permute y_train) | MAE [°C]: 0.256902277469635 | RMSE [°C]: 0.3200617730617523
TEST B (X_test = 0) | MAE [°C]: 0.261169970035553 | RMSE [°C]: 0.319918692111969
TEST C (shift window by +1) | MAE [°C]: 0.2768901288509369 | RMSE [°C]: 0.32125940918922424


In [44]:
# Foreword  # sprawdzamy czy LSTM faktycznie coś wnosi: baseline "persistence", skala ΔT3, oraz wersja h-step ahead (H=10) # sanity suite
# Why  # jeśli persistence ~= model, to model nic nie uczy z wejść; h-step wymusza korzystanie z T1/F1/T2/F2/F3 # rationale

import numpy as np  # numeryka # np
import plotly.graph_objects as go  # wykresy # go

# ---------- 1) BASELINE: persistence (T3_hat(t)=T3(t-1)) na tym samym odcinku testu ---------- # baseline
test_start_idx = int(n_train + n_val)  # start okien testu w X_seq/y_seq # start
n_test_windows = X_test_std.shape[0]  # liczba okien test # n_test

T3_prev_test = y_raw[(test_start_idx + T_window - 1) : (test_start_idx + T_window - 1 + n_test_windows)].astype(np.float32)  # T3(t-1) # prev
T3_true_test = y_raw[(test_start_idx + T_window)     : (test_start_idx + T_window     + n_test_windows)].astype(np.float32)  # T3(t) # true

T3_pred_persist = T3_prev_test.copy()  # persistence # persist
err_persist = T3_true_test - T3_pred_persist  # błąd # err

mae_persist = float(np.mean(np.abs(err_persist)))  # MAE # mae
rmse_persist = float(np.sqrt(np.mean(err_persist**2)))  # RMSE # rmse

print("PERSISTENCE baseline | MAE [°C]:", mae_persist, "| RMSE [°C]:", rmse_persist)  # print # print

# ---------- 2) SKALA ΔT3: czy to jest 'prawie zero'? ---------- # delta scale
dT3_all = (y_raw[1:] - y_raw[:-1]).astype(np.float32)  # ΔT3 per próbka # delta
print("ΔT3 stats [°C/step] | mean:", float(dT3_all.mean()), "| std:", float(dT3_all.std()), "| p95(|Δ|):", float(np.quantile(np.abs(dT3_all), 0.95)))  # stats # stats

# ---------- 3) H-STEP AHEAD dataset (H=10): target = T3(t+H) ---------- # h-step
H = 10  # horyzont predykcji (zmień na 5/10/20) # H

N = len(y_raw)  # N # N
NwH = N - T_window - H + 1  # liczba okien dla h-step # N windows H

X_seq_H = np.zeros((NwH, T_window, len(feature_cols)), dtype=np.float32)  # X okna # X
y_seq_H = np.zeros((NwH,), dtype=np.float32)  # y = T3(t+H) # y

for i in range(NwH):  # loop # loop
    X_seq_H[i] = X_raw[i:i+T_window]  # okno # window
    y_seq_H[i] = y_raw[i+T_window+H-1]  # target H kroków do przodu # target

# split czasowy jak wcześniej (proporcje) # split
n_train_H = int(NwH * train_frac)  # train # train
n_val_H = int(NwH * val_frac)  # val # val

X_train_H = X_seq_H[:n_train_H]  # train # train
y_train_H = y_seq_H[:n_train_H]  # train # train
X_val_H = X_seq_H[n_train_H:n_train_H+n_val_H]  # val # val
y_val_H = y_seq_H[n_train_H:n_train_H+n_val_H]  # val # val
X_test_H = X_seq_H[n_train_H+n_val_H:]  # test # test
y_test_H = y_seq_H[n_train_H+n_val_H:]  # test # test

# normalizacja X (fit na train) # X scale
X_train_H_flat = X_train_H.reshape(-1, X_train_H.shape[-1])  # flat # flat
X_mean_H = X_train_H_flat.mean(axis=0)  # mean # mean
X_std_H = X_train_H_flat.std(axis=0) + 1e-8  # std # std

X_train_H_std = (X_train_H - X_mean_H) / X_std_H  # train # train
X_val_H_std   = (X_val_H   - X_mean_H) / X_std_H  # val # val
X_test_H_std  = (X_test_H  - X_mean_H) / X_std_H  # test # test

# normalizacja y (fit na train) # y scale
y_mean_H = y_train_H.mean()  # mean # mean
y_std_H  = y_train_H.std() + 1e-8  # std # std

y_train_H_std = (y_train_H - y_mean_H) / y_std_H  # train # train
y_val_H_std   = (y_val_H   - y_mean_H) / y_std_H  # val # val
y_test_H_std  = (y_test_H  - y_mean_H) / y_std_H  # test # test

print("H-step dataset | X_test_H_std:", X_test_H_std.shape, "| y_test_H_std:", y_test_H_std.shape)  # shapes # shapes

# ---------- 4) H-STEP baseline: persistence-H (T3_hat(t+H)=T3(t)) ---------- # baseline H
# Dla okna i: "teraz" odpowiada indeksowi i+T_window-1, a target to i+T_window+H-1 # align
test_start_idx_H = int(n_train_H + n_val_H)  # start okien testu H # start
n_test_H = X_test_H_std.shape[0]  # n # n

T3_now_H = y_raw[(test_start_idx_H + T_window - 1) : (test_start_idx_H + T_window - 1 + n_test_H)].astype(np.float32)  # T3(t) # now
T3_true_H = y_raw[(test_start_idx_H + T_window + H - 1) : (test_start_idx_H + T_window + H - 1 + n_test_H)].astype(np.float32)  # T3(t+H) # true

T3_pred_persist_H = T3_now_H.copy()  # persistence-H # persistH
err_persist_H = T3_true_H - T3_pred_persist_H  # err # err

mae_persist_H = float(np.mean(np.abs(err_persist_H)))  # mae # mae
rmse_persist_H = float(np.sqrt(np.mean(err_persist_H**2)))  # rmse # rmse

print(f"PERSISTENCE-H baseline (H={H}) | MAE [°C]:", mae_persist_H, "| RMSE [°C]:", rmse_persist_H)  # print # print

# ---------- 5) (opcjonalnie) wykres: ΔT3 histogram (żeby zobaczyć łatwość zadania) ---------- # plot delta
fig = go.Figure()  # fig # fig
fig.add_trace(go.Histogram(x=dT3_all, nbinsx=80, name="ΔT3 per step"))  # hist # hist
fig.update_layout(template="plotly_dark", title="ΔT3 distribution (per time step)", xaxis_title="ΔT3 [°C/step]", yaxis_title="count")  # layout # layout
fig.show()  # show # show


PERSISTENCE baseline | MAE [°C]: 0.2717770040035248 | RMSE [°C]: 0.33613795042037964
ΔT3 stats [°C/step] | mean: -0.009313207119703293 | std: 0.4177361726760864 | p95(|Δ|): 0.9782087802886963
H-step dataset | X_test_H_std: (187, 60, 5) | y_test_H_std: (187,)
PERSISTENCE-H baseline (H=10) | MAE [°C]: 2.6912648677825928 | RMSE [°C]: 3.2968015670776367


In [45]:
# Foreword  # trenujemy LSTM na H-step ahead (H=10): wejście okno (T=60, 5 cech) -> wyjście T3(t+H) # H-step LSTM
# Why  # 1-step jest banalny (persistence wygrywa), H-step wymusza użycie wejść i dynamiki # rationale

import numpy as np  # numeryka # np
import tensorflow as tf  # TF # tf
from tensorflow.keras import Sequential  # model # Sequential
from tensorflow.keras.layers import Input, LSTM, Dense  # warstwy # layers
from tensorflow.keras.optimizers import Adam  # optymalizator # Adam
import plotly.graph_objects as go  # wykresy # go

assert "X_train_H_std" in globals(), "Brak X_train_H_std - najpierw zbuduj H-step dataset (komórka wcześniej)"  # guard # guard
assert "y_train_H_std" in globals(), "Brak y_train_H_std - najpierw zbuduj H-step y (std)"  # guard # guard
assert "X_val_H_std" in globals() and "y_val_H_std" in globals(), "Brak walidacji H-step"  # guard # guard
assert "X_test_H_std" in globals() and "y_test_H_std" in globals(), "Brak testu H-step"  # guard # guard
assert "y_mean_H" in globals() and "y_std_H" in globals(), "Brak y_mean_H/y_std_H do inverse"  # guard # guard
assert "T_window" in globals() and "H" in globals(), "Brak T_window/H"  # guard # guard
assert "tank_raw_df" in globals() and "y_raw" in globals(), "Brak tank_raw_df/y_raw do osi czasu i true"  # guard # guard
assert "n_train_H" in globals() and "n_val_H" in globals(), "Brak n_train_H/n_val_H do align"  # guard # guard

# --- hiperparametry do późniejszego tuningu (zmieniaj tu, nie w 20 miejscach) # hparams
HP_LSTM_UNITS = 32  # units # units
HP_LR = 3e-4  # learning rate # lr
HP_BATCH = 32  # batch # batch
HP_EPOCHS = 150  # max epochs # epochs
HP_CLIPNORM = 1.0  # clipnorm # clipnorm

# --- model (minimalny, sekwencyjny) # model
hstep_model = Sequential(name=f"tank_lstm_h{H}_T{T_window}")  # model # model
hstep_model.add(Input(shape=(T_window, X_train_H_std.shape[-1])))  # input # input
hstep_model.add(LSTM(HP_LSTM_UNITS))  # LSTM # LSTM
hstep_model.add(Dense(1, activation="linear"))  # output # out

hstep_model.compile(  # compile # compile
    optimizer=Adam(learning_rate=HP_LR, clipnorm=HP_CLIPNORM),  # Adam + clip # opt
    loss="mse",  # mse # mse
    metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],  # mae # mae
)

cb = [  # callbacks # callbacks
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),  # ES # ES
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-5),  # RLR # RLR
]  # end # end

hist_h = hstep_model.fit(  # fit # fit
    X_train_H_std, y_train_H_std,  # train # train
    validation_data=(X_val_H_std, y_val_H_std),  # val # val
    epochs=HP_EPOCHS,  # epochs # epochs
    batch_size=HP_BATCH,  # batch # batch
    callbacks=cb,  # cb # cb
    verbose=1,  # verbose # verbose
)

# --- predykcja i inverse do °C # predict
y_pred_H_std = hstep_model.predict(X_test_H_std, verbose=0).reshape(-1)  # pred std # pred
y_pred_H_C = (y_pred_H_std * float(y_std_H) + float(y_mean_H)).astype(np.float32)  # inverse # inv

y_true_H_C = (y_test_H_std * float(y_std_H) + float(y_mean_H)).astype(np.float32)  # inverse true # true
err_H = y_true_H_C - y_pred_H_C  # error # err

mae_H = float(np.mean(np.abs(err_H)))  # mae # mae
rmse_H = float(np.sqrt(np.mean(err_H**2)))  # rmse # rmse
print(f"LSTM H-step (H={H}) | MAE [°C]:", mae_H, "| RMSE [°C]:", rmse_H)  # print # print

# --- baseline persistence-H (T3_hat(t+H)=T3(t)) do porównania # baseline H
test_start_idx_H = int(n_train_H + n_val_H)  # start okien testu H # start
n_test_H = X_test_H_std.shape[0]  # liczba okien test # n

T3_now_H = y_raw[(test_start_idx_H + T_window - 1) : (test_start_idx_H + T_window - 1 + n_test_H)].astype(np.float32)  # T3(t) # now
T3_true_H = y_raw[(test_start_idx_H + T_window + H - 1) : (test_start_idx_H + T_window + H - 1 + n_test_H)].astype(np.float32)  # T3(t+H) # true

err_persist_H = T3_true_H - T3_now_H  # err # err
mae_persist_H = float(np.mean(np.abs(err_persist_H)))  # mae # mae
rmse_persist_H = float(np.sqrt(np.mean(err_persist_H**2)))  # rmse # rmse
print(f"PERSISTENCE-H baseline (H={H}) | MAE [°C]:", mae_persist_H, "| RMSE [°C]:", rmse_persist_H)  # print # print

# --- wykres true vs pred (na osi czasu) # plot
t_all = tank_raw_df["t_s"].values.astype(np.float32)  # czas # time
t_test_H = t_all[(test_start_idx_H + T_window + H - 1) : (test_start_idx_H + T_window + H - 1 + n_test_H)]  # align time # t

fig = go.Figure()  # fig # fig
fig.add_trace(go.Scatter(x=t_test_H, y=T3_true_H, mode="lines", name="T3_true_C"))  # true # true
fig.add_trace(go.Scatter(x=t_test_H, y=y_pred_H_C, mode="lines", name=f"T3_pred_C (LSTM H={H})"))  # pred # pred
fig.add_trace(go.Scatter(x=t_test_H, y=T3_now_H, mode="lines", name=f"T3_persist_C (baseline)", line=dict(dash="dot")))  # baseline # base
fig.update_layout(  # layout # layout
    template="plotly_dark",  # dark # dark
    title=f"TEST | T3 true vs pred | H-step={H} (window-based)",  # title # title
    xaxis_title="t [s]",  # x # x
    yaxis_title="T3 [°C]",  # y # y
)
fig.show()  # show # show

# --- wykres residuals (LSTM vs baseline) # residuals
fig_e = go.Figure()  # fig # fig
fig_e.add_trace(go.Scatter(x=t_test_H, y=(T3_true_H - y_pred_H_C), mode="lines", name="err_LSTM = true - pred"))  # err lstm # err
fig_e.add_trace(go.Scatter(x=t_test_H, y=err_persist_H, mode="lines", name="err_baseline = true - persist", line=dict(dash="dot")))  # err base # base
fig_e.update_layout(template="plotly_dark", title=f"TEST | residuals | H-step={H}", xaxis_title="t [s]", yaxis_title="error [°C]")  # layout # layout
fig_e.show()  # show # show


Epoch 1/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - loss: 0.9139 - mae: 0.8872 - val_loss: 0.6568 - val_mae: 0.7671 - learning_rate: 3.0000e-04
Epoch 2/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.7056 - mae: 0.7699 - val_loss: 0.6214 - val_mae: 0.7496 - learning_rate: 3.0000e-04
Epoch 3/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.4613 - mae: 0.6140 - val_loss: 0.5772 - val_mae: 0.7196 - learning_rate: 3.0000e-04
Epoch 4/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.2941 - mae: 0.4711 - val_loss: 0.4963 - val_mae: 0.6540 - learning_rate: 3.0000e-04
Epoch 5/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.1408 - mae: 0.3066 - val_loss: 0.3872 - val_mae: 0.5463 - learning_rate: 3.0000e-04
Epoch 6/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.0745 - mae: 0.2156 - val_loss: 0.2728 - val_mae: 0.4219 - learning_rate: 3.0000e-04
Epoch 7/150
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.0494 - mae: 0.1683 - val_loss: 0.1889 - val_mae: 0.3219 - lear

In [48]:
# Foreword  # online 1-step predykcja: w każdej chwili bierzemy ostatnie T próbek wejść i przewidujemy T3 na kolejny krok # online
# Why  # okno ma tylko realną historię, więc to jest uczciwy "feedback" na procesie # rationale

model_1step = model  # <- ZMIEŃ na Twoją nazwę: np. tank_model / hstep_model / best_model # pick model
T = 60  # okno # T
feature_cols = ["T1_C","F1_m3s","T2_C","F2_m3s","F3_m3s"]  # wejścia # features

t = tank_raw_df["t_s"].values  # czas # t
y_true = tank_raw_df["T3_C"].values  # true # y
y_pred = np.full_like(y_true, np.nan, dtype=float)  # pred buf # pred

for k in range(T, len(tank_raw_df)):  # loop po czasie # loop
    Xw = tank_raw_df.loc[k-T:k-1, feature_cols].values.astype(float)  # okno wejść # window
    Xw = (Xw - X_mean) / X_std  # skala jak w treningu # scale
    yhat_std = model_1step.predict(Xw.reshape(1, T, len(feature_cols)), verbose=0)[0,0]  # pred std # pred
    y_pred[k] = yhat_std * y_std + y_mean  # inverse do °C # inv

mask = ~np.isnan(y_pred)  # gdzie mamy pred # mask
err = y_true[mask] - y_pred[mask]  # error # err
print("ONLINE 1-step | MAE [°C]:", float(np.mean(np.abs(err))), "| RMSE [°C]:", float(np.sqrt(np.mean(err**2))))  # metrics # metrics

fig = go.Figure()  # fig # fig
fig.add_trace(go.Scatter(x=t[mask], y=y_true[mask], mode="lines", name="T3_true_C"))  # true # true
fig.add_trace(go.Scatter(x=t[mask], y=y_pred[mask], mode="lines", name="T3_pred_C (online 1-step)"))  # pred # pred
fig.update_layout(template="plotly_dark", title=f"ONLINE 1-step | window T={T}", xaxis_title="t [s]", yaxis_title="T3 [°C]")  # layout # layout
fig.show()  # show # show


ONLINE 1-step | MAE [°C]: 7.830583348017673 | RMSE [°C]: 9.627649111595355
